# Финальная оценка и error analysis

После выбора V2 на validation оцениваем её один раз на нетронутом test и разбираем наиболее частые ошибки

Веса модели не входят в репозиторий: локально ожидается `models/v2_weighted`

## 1. Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from collections import Counter
from datasets import load_dataset, ClassLabel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from src.project_utils import (
    SEED, MODEL_NAME, DATASET_NAME, NUM_LABELS, MAX_LENGTH,
    set_seed, clean_html, build_text, build_text_with_attributes,
    compute_metrics, make_model, WeightedTrainer, predict_category,
)

set_seed(SEED)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


c:\Users\alive\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.14.0+cu126
CUDA: True


In [2]:
dataset = load_dataset(DATASET_NAME)
print(dataset)
print(dataset["train"].column_names)
print("Rows:", dataset["train"].num_rows)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'category', 'parent_category', 'grandparent_category', 'greatgrandparent_category', 'rating', 'description', 'comments', 'attributes', 'seller', 'seller_description', 'adultCategory'],
        num_rows: 2059468
    })
})
['id', 'title', 'category', 'parent_category', 'grandparent_category', 'greatgrandparent_category', 'rating', 'description', 'comments', 'attributes', 'seller', 'seller_description', 'adultCategory']
Rows: 2059468


In [3]:
target_column = "greatgrandparent_category"
none_count = sum(value is None for value in dataset["train"][target_column])
print("Total:", dataset["train"].num_rows)
print("Missing target:", none_count)
print("Valid target:", dataset["train"].num_rows - none_count)

Total: 2059468
Missing target: 657606
Valid target: 1401862


In [4]:
dataset = dataset.filter(lambda example: example[target_column] is not None)
label_names = sorted(set(dataset["train"][target_column]))
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}
print("Classes:", len(label_names))

Classes: 39


In [5]:
dataset = dataset.map(build_text)
dataset = dataset.map(lambda example: {"labels": label2id[example[target_column]]})
dataset = dataset.cast_column("labels", ClassLabel(num_classes=len(label_names), names=label_names))
print(dataset["train"][0]["text"])

Свитер вязанный осень-зима 2017 Очень хороший свитер


In [6]:
train_test = dataset["train"].train_test_split(test_size=0.10, seed=SEED, stratify_by_column="labels")
val_test = train_test["test"].train_test_split(test_size=0.5, seed=SEED, stratify_by_column="labels")
train_dataset = train_test["train"]
val_dataset = val_test["train"]
test_dataset = val_test["test"]
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 1261675
Validation: 70093
Test: 70094


In [12]:
model_columns = ["input_ids", "attention_mask", "labels"]
def keep_model_columns(split):
    split = split.remove_columns([c for c in split.column_names if c not in model_columns])
    split.set_format("torch", columns=model_columns)
    return split

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
test_model_dataset = keep_model_columns(test_dataset.map(tokenize, batched=True))
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
V2_DIR = Path("../models/v2_weighted")

## 6. Final test

In [14]:
final_model = AutoModelForSequenceClassification.from_pretrained(V2_DIR)
final_trainer = Trainer(
    model=final_model,
    args=TrainingArguments(output_dir="./final_eval", per_device_eval_batch_size=128, report_to="none"),
    processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
)
final_test_metrics = final_trainer.evaluate(test_model_dataset)
final_test_metrics

Loading weights: 100%|██████████| 57/57 [00:00<00:00, 7111.53it/s]


Training Loss,Validation Loss,Step,Accuracy,F1 Macro,F1 Weighted
No log,0.148575,0,0.973364,0.949350,0.973361


{'eval_loss': 0.1485750675201416,
 'eval_accuracy': 0.9733643393157759,
 'eval_f1_macro': 0.9493495307927885,
 'eval_f1_weighted': 0.9733610631479912}

### Зафиксированные результаты

| Metric | Test |
|---|---:|
| Accuracy | **0.9734** |
| Macro F1 | **0.9493** |
| Weighted F1 | **0.9734** |

## 7. Error analysis

In [15]:
test_predictions = final_trainer.predict(test_model_dataset)
test_pred_ids = np.argmax(test_predictions.predictions, axis=1)
test_true_ids = test_predictions.label_ids
cm = confusion_matrix(test_true_ids, test_pred_ids)

confusions = []
for true_id in range(NUM_LABELS):
    total_true = cm[true_id].sum()
    total_errors = total_true - cm[true_id, true_id]
    for pred_id in range(NUM_LABELS):
        if true_id == pred_id:
            continue
        errors = cm[true_id, pred_id]
        if errors:
            confusions.append({
                "true_class": label_names[true_id],
                "predicted_class": label_names[pred_id],
                "errors": int(errors),
                "error_rate": errors / total_true,
                "share_of_errors": errors / total_errors if total_errors else 0,
            })

confusions_df = pd.DataFrame(confusions).sort_values("errors", ascending=False).reset_index(drop=True)
confusions_df.head(15)

,true_class,predicted_class,errors,error_rate,share_of_errors
0,Детская одежда,Одежда,134,0.063810,0.930556
1,Одежда,Детская одежда,113,0.014232,0.837037
2,Товары для дома,Товары для кухни,73,0.010680,0.271375
3,Товары для кухни,Товары для дома,65,0.044828,0.747126
4,Хобби и творчество,Товары для дома,60,0.009336,0.307692
5,Товары для дома,Хобби и творчество,52,0.007608,0.193309
6,Хобби и творчество,Аксессуары,49,0.007624,0.251282
7,Аксессуары,Хобби и творчество,45,0.004720,0.263158
8,Аксессуары,Женские аксессуары,45,0.004720,0.263158
9,Товары для дома,Детские товары,26,0.003804,0.096654


### Основные наблюдения

Ошибки в основном возникают между семантически близкими категориями: `Детская одежда <-> Одежда`, `Товары для дома <-> Товары для кухни`, `Хобби и творчество <-> Товары для дома`, `Аксессуары <-> Женские аксессуары`. Это указывает прежде всего на близость самих товарных категорий, а не на случайные ошибки